# Modélisation des données & Load — Démo

Ce notebook teste le schéma (Option A, 6 tables) construit collectivement en atelier, d'abord avec des données d'exemple, puis avec un vrai extrait de vos propres données transformées (`films_transformes.json`).

## 1. Rappel du besoin

Vous êtes à l'étape **Load** du pipeline ETL : Extract et Transform sont déjà faits, votre `data/processed/films_transformes.json` contient déjà films + genres + équipe + tags + enrichissements Wikipedia, fusionnés. Ce notebook teste le schéma retenu avant de l'appliquer à vos propres données.

In [1]:
import json
import sqlite3
import pandas as pd

# Base en mémoire pour cette démo (rien n'est écrit sur disque ici)
connexion = sqlite3.connect(":memory:")
connexion.execute("PRAGMA foreign_keys = ON")

## 2. Créer les tables

`films` (avec les champs TMDB + Wikipedia + agrégats MovieLens), `genres`/`film_genre` (N-N), `personnes`/`film_personne` (N-N avec `role` + `personnage` portés par la relation), `film_tag` (1-N).

In [2]:
with open("../sql/schema.sql", "r", encoding="utf-8") as f:
    script_sql = f.read()

connexion.executescript(script_sql)
print("Tables créées.")

Tables créées.


## 3. Insérer un exemple à la main

Un film, deux genres, un réalisateur et un acteur avec son personnage, un tag — pour observer chaque relation avant de charger de vraies données.

In [3]:
connexion.execute(
    "INSERT INTO films (id, title, release_date, vote_average) VALUES (?, ?, ?, ?)",
    (1, "Exemple de film", "2026-01-01", 7.5),
)

connexion.execute("INSERT INTO genres (nom) VALUES ('Action')")
connexion.execute("INSERT INTO genres (nom) VALUES ('Science-Fiction')")
connexion.execute("INSERT INTO film_genre (film_id, genre_id) VALUES (1, 1)")
connexion.execute("INSERT INTO film_genre (film_id, genre_id) VALUES (1, 2)")

connexion.execute(
    "INSERT INTO personnes (id, name) VALUES (101, 'Une Réalisatrice')"
)
connexion.execute(
    "INSERT INTO film_personne (film_id, personne_id, role, personnage) VALUES (1, 101, 'Réalisateur', NULL)"
)

connexion.execute(
    "INSERT INTO personnes (id, name) VALUES (102, 'Un Acteur')"
)
connexion.execute(
    "INSERT INTO film_personne (film_id, personne_id, role, personnage) VALUES (1, 102, 'Acteur', 'Le héros')"
)

connexion.execute("INSERT INTO film_tag (film_id, tag) VALUES (1, 'voyage dans le temps')")

connexion.commit()
print("Exemple inséré.")

Exemple inséré.


## 4. Vérifier la relation avec attribut

Le `personnage` n'existe que pour la ligne "Acteur" — c'est bien un attribut de la relation, pas de la personne (une réalisatrice n'a pas de personnage joué).

In [4]:
requete = """
SELECT films.title, personnes.name, film_personne.role, film_personne.personnage
FROM film_personne
JOIN films ON films.id = film_personne.film_id
JOIN personnes ON personnes.id = film_personne.personne_id
"""

pd.read_sql_query(requete, connexion)

,title,name,role,personnage
0,Exemple de film,Une Réalisatrice,Réalisateur,NaN
1,Exemple de film,Un Acteur,Acteur,Le héros


## 5. Charger un vrai extrait de vos données transformées

On repart d'une base vierge et on charge cette fois quelques films de votre `films_transformes.json` réel, avec les mêmes fonctions que `db.py`/`load.py`.

In [5]:
import sys
sys.path.append("../src")
from db import (
    creer_connexion,
    creer_tables,
    inserer_film,
    associer_film_genre,
    associer_film_personne,
    inserer_tag,
    charger_json,
)

with open("../data/processed/films_transformes.json", "r", encoding="utf-8") as f:
    films_transformes = json.load(f)

# Un petit sous-ensemble suffit pour vérifier que tout s'enchaîne correctement
echantillon = films_transformes[:5]
print(f"{len(echantillon)} film(s) chargé(s) pour ce test, sur {len(films_transformes)} disponibles.")

5 film(s) chargé(s) pour ce test, sur 196 disponibles.


In [7]:
connexion_reelle = creer_connexion(":memory:")
creer_tables(connexion_reelle, chemin_schema="../sql/schema.sql")

for film in echantillon:
    inserer_film(connexion_reelle, film)
    for nom_genre in film.get("genres", []):
        associer_film_genre(connexion_reelle, film["id"], nom_genre)
    for personne in film.get("equipe", []):
        associer_film_personne(connexion_reelle, film["id"], personne)
    for tag in film.get("tags", []):
        inserer_tag(connexion_reelle, film["id"], tag)

connexion_reelle.commit()
print("Chargement terminé.")

Tables créées (ou déjà existantes).
Chargement terminé.


In [8]:
pd.read_sql_query(
    "SELECT title, release_date, note_moyenne_movielens, musique FROM films",
    connexion_reelle,
)

,title,release_date,note_moyenne_movielens,musique
0,Spider-Man: Brand New Day,2026-07-29,None,Michael Giacchino
1,The End of Oak Street,2026-08-12,None,Michael Giacchino
2,Coyote vs. Acme,2026-08-20,None,Steven Price
3,The Odyssey,2026-07-15,None,Ludwig Göransson
4,Resident Evil,2026-09-16,None,Hays Holladay Ryan Holladay


---
**À vous :** adaptez votre propre `schema.sql` (identique, inspiré, ou différent de celui-ci — voir la présentation en atelier), puis chargez l'intégralité de votre `films_transformes.json` avec votre propre script, sur le modèle de cette dernière cellule.